# Notebook 02: Actuarial Benchmarking (Li-Lee, Sex-Specific)

## Overview
This notebook implements the **Li-Lee (2005) multi-population mortality model** separately for Male and Female populations. It serves as the actuarial baseline against which the AINN challenger model will be benchmarked.

All extracted parameters are persisted to disk so that downstream notebooks can load them directly without re-execution.

## Objectives
1. **Lee-Carter (Independent)**: Extract $a_x$, $b_x$, $k_t$ via SVD for each country and sex.
2. **Li-Lee Common Factor**: Compute the shared mortality trend ($K_t$, $B_x$) from the cluster average, separately for Male and Female.
3. **Country-Specific Factors**: Extract residual factors ($k_{t,i}$) capturing local deviations from the common trend.
4. **Stationarity Analysis**: ADF and KPSS tests on specific factors to assess Li-Lee coherence assumptions by sex.
5. **Persistence**: Save all parameters (pickle + CSV) for downstream use.

## 2.1: Setup & Data Loading

In [ ]:
import sys
sys.path.append('../src')
from reproducibility import set_seed
set_seed(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json as json_lib
import os
import warnings
warnings.filterwarnings('ignore')

from scipy.linalg import svd
from statsmodels.tsa.stattools import adfuller, kpss

from style_config import set_style, save_dual, COUNTRIES, COUNTRY_COLORS

set_style("notebook")

# Paths
PROCESSED_DIR = "../data/processed/"
FIGURES_DIR = "../reports/figures/"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load metadata
with open(os.path.join(PROCESSED_DIR, "metadata.json"), "r") as f:
    meta = json_lib.load(f)

YEARS = np.array(meta["years"])
AGES = np.array(meta["ages"])
N_YEARS = len(YEARS)
N_AGES = len(AGES)
N_COUNTRIES = len(COUNTRIES)

print(f"Loaded metadata: {N_COUNTRIES} countries, {N_AGES} ages, {N_YEARS} years")
print(f"Years: {YEARS[0]}-{YEARS[-1]}, Ages: {AGES[0]}-{AGES[-1]}")

# Load log-mortality matrices
log_matrices = {}
for code in COUNTRIES:
    log_matrices[code] = {}
    for sex in ["male", "female"]:
        path = os.path.join(PROCESSED_DIR, f"{code}_log_mx_{sex}.npy")
        log_matrices[code][sex] = np.load(path)

print(f"\nLoaded {N_COUNTRIES} x 2 sexes = {N_COUNTRIES * 2} log-mortality matrices")
print(f"Shape: {log_matrices['CHE']['male'].shape} (ages x years)")

## 2.2: Lee-Carter Independent Model (SVD)

For each country $i$ and sex $s$, we decompose:
$$\ln(m_{x,t,i,s}) = a_{x,i,s} + b_{x,i,s} \cdot k_{t,i,s} + \epsilon_{x,t,i,s}$$

Parameters extracted via Singular Value Decomposition on the centred matrix.

In [ ]:
def fit_lee_carter(log_mx_matrix):
    """
    Fit Lee-Carter model via SVD.
    
    Args:
        log_mx_matrix: np.ndarray of shape (n_ages, n_years), containing ln(m_x,t).
    
    Returns:
        dict with keys: ax, bx, kt
    """
    # ax: mean log-mortality over time for each age
    ax = log_mx_matrix.mean(axis=1)
    
    # Centre the matrix
    centred = log_mx_matrix - ax[:, np.newaxis]
    
    # SVD: rank-1 approximation
    U, S, Vt = svd(centred, full_matrices=False)
    
    # First component
    bx_raw = U[:, 0]
    kt_raw = S[0] * Vt[0, :]
    
    # Identifiability constraint: sum(bx) = 1
    bx = bx_raw / bx_raw.sum()
    kt = kt_raw * bx_raw.sum()
    
    return {"ax": ax, "bx": bx, "kt": kt}


# Fit Lee-Carter for all countries and both sexes
lc_params = {}
for code in COUNTRIES:
    lc_params[code] = {}
    for sex in ["male", "female"]:
        lc_params[code][sex] = fit_lee_carter(log_matrices[code][sex])

print("Lee-Carter parameters extracted for all countries and sexes.")
print(f"Example (CHE, male): ax shape={lc_params['CHE']['male']['ax'].shape}, "
      f"bx shape={lc_params['CHE']['male']['bx'].shape}, "
      f"kt shape={lc_params['CHE']['male']['kt'].shape}")

## 2.3: Mortality Index $k_t$ (Independent Lee-Carter)

Visual comparison of the time trend across the cluster, by sex.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for sex_idx, sex in enumerate(["male", "female"]):
    ax = axes[sex_idx]
    for code, name in COUNTRIES.items():
        ax.plot(YEARS, lc_params[code][sex]["kt"],
                color=COUNTRY_COLORS[code], label=name, linewidth=1.5)
    ax.set_title(f"{sex.capitalize()}")
    ax.set_xlabel("Year")
    if sex_idx == 0:
        ax.set_ylabel("$k_t$ (Mortality Index)")
    ax.legend(loc="upper right", framealpha=0.9)
    ax.axvline(x=2011, color="grey", linestyle="--", alpha=0.4)

fig.suptitle("Independent Lee-Carter: Mortality Index by Sex", fontsize=14, y=1.02)
plt.tight_layout()
save_dual(fig, "fig04_lc_kt_independent_by_sex")
plt.show()

## 2.4: Li-Lee Common Factor Extraction

The Li-Lee model decomposes mortality into a **Common Factor** shared by the cluster and **Specific Factors** for each country:
$$\ln(m_{x,t,i}) = a_{x,i} + B_x K_t + b_{x,i} k_{t,i} + \epsilon_{x,t,i}$$

We compute $K_t$ and $B_x$ from the average log-mortality matrix across all countries, separately for each sex.

In [ ]:
def fit_li_lee_common(log_matrices_dict, sex):
    """
    Extract Li-Lee Common Factor from the cluster average.
    
    Args:
        log_matrices_dict: dict {country_code: {sex: matrix}}
        sex: "male" or "female"
    
    Returns:
        dict with keys: Kt, Bx, Ax_common
    """
    # Average log-mortality across all countries
    all_matrices = np.array([log_matrices_dict[code][sex] for code in COUNTRIES])
    average_matrix = all_matrices.mean(axis=0)  # shape: (n_ages, n_years)
    
    # Common age profile
    Ax_common = average_matrix.mean(axis=1)
    
    # Centre and SVD
    centred = average_matrix - Ax_common[:, np.newaxis]
    U, S, Vt = svd(centred, full_matrices=False)
    
    # First component with constraint sum(Bx) = 1
    Bx_raw = U[:, 0]
    Kt_raw = S[0] * Vt[0, :]
    
    Bx = Bx_raw / Bx_raw.sum()
    Kt = Kt_raw * Bx_raw.sum()
    
    return {"Kt": Kt, "Bx": Bx, "Ax_common": Ax_common}


# Extract common factors for both sexes
common_factors = {}
for sex in ["male", "female"]:
    common_factors[sex] = fit_li_lee_common(log_matrices, sex)

print("Li-Lee Common Factors extracted.")
for sex in ["male", "female"]:
    print(f"  {sex.capitalize()}: Kt range [{common_factors[sex]['Kt'].min():.2f}, "
          f"{common_factors[sex]['Kt'].max():.2f}]")

## 2.5: Common Factor $K_t$ (Male vs Female)

In [ ]:
fig, ax = plt.subplots()

ax.plot(YEARS, common_factors["male"]["Kt"], color=COUNTRY_COLORS["CHE"],
        label="Male", linewidth=2)
ax.plot(YEARS, common_factors["female"]["Kt"], color=COUNTRY_COLORS["JPN"],
        label="Female", linewidth=2, linestyle="--")

ax.set_xlabel("Year")
ax.set_ylabel("$K_t$ (Common Mortality Index)")
ax.set_title("Li-Lee Common Factor: Male vs Female")
ax.legend()
ax.axvline(x=2011, color="grey", linestyle="--", alpha=0.4)

plt.tight_layout()
save_dual(fig, "fig05_li_lee_common_factor_sex")
plt.show()

## 2.6: Country-Specific Factors $k_{t,i}$

Residuals after removing the common trend, extracted via a second SVD on the country-level residual matrix.

In [ ]:
def extract_specific_factors(log_matrices_dict, common_factors_dict, sex):
    """
    Extract country-specific factors by removing the common trend.
    
    For each country i:
        residual_matrix = ln(m_x,t,i) - a_x,i - Bx * Kt
    Then SVD on the residual to get bx_i, kt_i.
    
    Returns:
        dict {country_code: {"bx_specific": ..., "kt_specific": ...}}
    """
    Bx = common_factors_dict[sex]["Bx"]
    Kt = common_factors_dict[sex]["Kt"]
    
    specific_params = {}
    for code in COUNTRIES:
        matrix = log_matrices_dict[code][sex]
        ax_i = matrix.mean(axis=1)
        
        # Remove common component
        residual = matrix - ax_i[:, np.newaxis] - np.outer(Bx, Kt)
        
        # SVD on residual
        U, S, Vt = svd(residual, full_matrices=False)
        
        bx_raw = U[:, 0]
        kt_raw = S[0] * Vt[0, :]
        
        # Constraint: sum(bx_i) = 1
        bx_specific = bx_raw / bx_raw.sum()
        kt_specific = kt_raw * bx_raw.sum()
        
        specific_params[code] = {
            "ax": ax_i,
            "bx_specific": bx_specific,
            "kt_specific": kt_specific
        }
    
    return specific_params


# Extract specific factors for both sexes
specific_factors = {}
for sex in ["male", "female"]:
    specific_factors[sex] = extract_specific_factors(log_matrices, common_factors, sex)

print("Country-specific factors extracted.")
for sex in ["male", "female"]:
    print(f"\n  {sex.capitalize()}:")
    for code, name in COUNTRIES.items():
        kt_s = specific_factors[sex][code]["kt_specific"]
        print(f"    {code}: kt_specific range [{kt_s.min():.3f}, {kt_s.max():.3f}]")

## 2.7: Country-Specific Factors $k_{t,i}$ by Sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for sex_idx, sex in enumerate(["male", "female"]):
    ax = axes[sex_idx]
    for code, name in COUNTRIES.items():
        kt_s = specific_factors[sex][code]["kt_specific"]
        ax.plot(YEARS, kt_s, color=COUNTRY_COLORS[code], label=name, linewidth=1.5)
    
    ax.axhline(y=0, color="grey", linestyle="-", alpha=0.3)
    ax.set_title(f"{sex.capitalize()}")
    ax.set_xlabel("Year")
    if sex_idx == 0:
        ax.set_ylabel("$k_{t,i}$ (Specific Factor)")
    ax.legend(loc="best", framealpha=0.9)

fig.suptitle("Li-Lee Country-Specific Factors by Sex", fontsize=14, y=1.02)
plt.tight_layout()
save_dual(fig, "fig06_li_lee_specific_factors_by_sex")
plt.show()

## 2.8: Stationarity Analysis (ADF + KPSS)

Li-Lee assumes country-specific factors are stationary. We test this assumption for each country and sex using both ADF ($H_0$: unit root) and KPSS ($H_0$: stationary).

In [ ]:
def stationarity_tests(series, name=""):
    """Run ADF and KPSS tests, return summary dict."""
    # ADF: H0 = unit root (non-stationary). Reject => stationary.
    adf_stat, adf_pval, *_ = adfuller(series, autolag='AIC')
    
    # KPSS: H0 = stationary. Reject => non-stationary.
    kpss_stat, kpss_pval, *_ = kpss(series, regression='c', nlags='auto')
    
    adf_pass = "PASS" if adf_pval < 0.05 else "FAIL"
    kpss_pass = "PASS" if kpss_pval > 0.05 else "FAIL"
    
    # Interpretation
    if adf_pass == "PASS" and kpss_pass == "PASS":
        status = "Stationary"
    elif adf_pass == "FAIL" and kpss_pass == "FAIL":
        status = "Unit Root"
    else:
        status = "Conflict"
    
    return {
        "Country": name,
        "ADF p-value": round(adf_pval, 4),
        "ADF": adf_pass,
        "KPSS p-value": round(kpss_pval, 4),
        "KPSS": kpss_pass,
        "Status": status
    }


# Run tests for all countries and both sexes
stationarity_results = {}
for sex in ["male", "female"]:
    results = []
    for code, name in COUNTRIES.items():
        kt_s = specific_factors[sex][code]["kt_specific"]
        res = stationarity_tests(kt_s, name=name)
        results.append(res)
    stationarity_results[sex] = pd.DataFrame(results)

# Display results
for sex in ["male", "female"]:
    print(f"\n{'='*60}")
    print(f"  STATIONARITY ANALYSIS: {sex.upper()}")
    print(f"{'='*60}")
    print(stationarity_results[sex].to_string(index=False))

## 2.9: First Differences ($\Delta K_t$, $\Delta k_{t,i}$)

Following Project 04, we transition to modelling **first differences** to handle non-stationarity. This is the input format for the AINN training.

In [ ]:
# Compute first differences for common and specific factors
delta_common = {}
for sex in ["male", "female"]:
    delta_common[sex] = np.diff(common_factors[sex]["Kt"])

delta_specific = {}
for sex in ["male", "female"]:
    delta_specific[sex] = {}
    for code in COUNTRIES:
        delta_specific[sex][code] = np.diff(specific_factors[sex][code]["kt_specific"])

print(f"First differences computed.")
print(f"Delta Kt length: {len(delta_common['male'])} (from {N_YEARS} years)")
print(f"Delta kt_i length: {len(delta_specific['male']['CHE'])}")

# Build the full feature matrix for AINN training: [delta_Kt, delta_kt_1, ..., delta_kt_6]
def build_feature_matrix(sex):
    """Build the 7-column feature matrix (1 common + 6 specific) for a given sex."""
    cols = [delta_common[sex]]
    col_names = ["delta_Kt"]
    for code in COUNTRIES:
        cols.append(delta_specific[sex][code])
        col_names.append(f"delta_kt_{code}")
    matrix = np.column_stack(cols)
    return matrix, col_names

feature_matrices = {}
feature_names = {}
for sex in ["male", "female"]:
    feature_matrices[sex], feature_names[sex] = build_feature_matrix(sex)
    print(f"\n{sex.capitalize()} feature matrix: shape = {feature_matrices[sex].shape}")
    print(f"  Columns: {feature_names[sex]}")

## 2.10: Persistence (Save All Parameters)

Save all extracted parameters so downstream notebooks can load them without re-executing this notebook.

In [ ]:
# Bundle everything into a single dict for persistence
li_lee_bundle = {
    "common_factors": common_factors,
    "specific_factors": specific_factors,
    "lc_params": lc_params,
    "delta_common": delta_common,
    "delta_specific": delta_specific,
    "feature_matrices": feature_matrices,
    "feature_names": feature_names,
    "stationarity_results": {sex: df.to_dict() for sex, df in stationarity_results.items()},
    "metadata": {
        "years": YEARS,
        "ages": AGES,
        "countries": list(COUNTRIES.keys()),
        "n_years": N_YEARS,
        "n_ages": N_AGES,
    }
}

# Save as pickle
output_path = os.path.join(PROCESSED_DIR, "li_lee_params.pkl")
with open(output_path, "wb") as f:
    pickle.dump(li_lee_bundle, f)

print(f"All Li-Lee parameters saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Also save feature matrices as .npy for fast loading in training notebooks
for sex in ["male", "female"]:
    np.save(os.path.join(PROCESSED_DIR, f"feature_matrix_{sex}.npy"), feature_matrices[sex])

print(f"\nFeature matrices saved as .npy (male + female)")
print(f"\nNotebook 02 complete. Proceed to Notebook 03 (Constrained Loss Design).")

## 2.11: Verification — Cross-Check with Project 04

Sanity check: run the same Li-Lee procedure on "Total" (both sexes combined) to verify consistency with Project 04 results.

In [ ]:
# === VERIFICATION CELL: Cross-check with Project 04 ===
# Run Li-Lee on "Total" (both sexes) to compare with Project 04 results
print("=" * 60)
print("  VERIFICATION: Li-Lee on Total (both sexes combined)")
print("=" * 60)

# Load Total matrices
log_matrices_total = {}
for code in COUNTRIES:
    path = os.path.join(PROCESSED_DIR, f"{code}_log_mx_total.npy")
    log_matrices_total[code] = np.load(path)

# Fit Li-Lee Common Factor on Total
all_total = np.array([log_matrices_total[code] for code in COUNTRIES])
avg_total = all_total.mean(axis=0)
Ax_total = avg_total.mean(axis=1)
centred_total = avg_total - Ax_total[:, np.newaxis]
U_t, S_t, Vt_t = svd(centred_total, full_matrices=False)
Bx_total = U_t[:, 0] / U_t[:, 0].sum()
Kt_total = (Vt_t[0, :] * S_t[0]) * U_t[:, 0].sum()

print(f"Kt (Total) range: [{Kt_total.min():.2f}, {Kt_total.max():.2f}]")
print(f"Kt (Male) range:  [{common_factors['male']['Kt'].min():.2f}, {common_factors['male']['Kt'].max():.2f}]")
print(f"Kt (Female) range:[{common_factors['female']['Kt'].min():.2f}, {common_factors['female']['Kt'].max():.2f}]")

# Extract specific factors on Total and run stationarity
print("\n--- Stationarity on TOTAL (should match Project 04) ---")
specific_total = {}
for code in COUNTRIES:
    matrix = log_matrices_total[code]
    ax_i = matrix.mean(axis=1)
    residual = matrix - ax_i[:, np.newaxis] - np.outer(Bx_total, Kt_total)
    U_s, S_s, Vt_s = svd(residual, full_matrices=False)
    bx_raw = U_s[:, 0]
    kt_raw = S_s[0] * Vt_s[0, :]
    kt_specific = kt_raw * bx_raw.sum()
    specific_total[code] = kt_specific

results_total = []
for code, name in COUNTRIES.items():
    res = stationarity_tests(specific_total[code], name=name)
    results_total.append(res)

df_total = pd.DataFrame(results_total)
print(df_total.to_string(index=False))

print("\n--- Project 04 reference (from RESEARCH_NOTES) ---")
print("CHE: Conflict (ADF FAIL 0.76, KPSS PASS)")
print("SWE: Unit Root (ADF FAIL 0.92, KPSS FAIL)")
print("NOR: Stationary (ADF PASS 0.00, KPSS PASS)")
print("DEUTW: Unit Root (ADF FAIL 0.94, KPSS FAIL)")
print("NLD: Unit Root (ADF FAIL 0.10, KPSS FAIL)")
print("JPN: Conflict (ADF PASS 0.04, KPSS FAIL)")
